In [43]:
import pandas as pd
import os
from pathlib import Path
import numpy as np
import warnings
import matplotlib.pyplot as plt
import seaborn as sns
from difflib import SequenceMatcher
import pandas as pd
from datetime import datetime, timedelta
from difflib import SequenceMatcher
import matplotlib.pyplot as plt
import os
from pathlib import Path
import seaborn as sns
import numpy as np
from lifelines.utils import to_long_format
from lifelines.utils import add_covariate_to_timeline
from lifelines import CoxTimeVaryingFitter
from statsmodels.distributions.empirical_distribution import ECDF
from scipy.stats import ks_2samp
from scipy import stats
from matplotlib.ticker import MultipleLocator
import matplotlib.ticker as mtick
import warnings


path_actual = Path.cwd()
path_data = path_actual.parent / 'Data'

In [54]:
estbs_not=[ 120103., 124130., 105107., 103102., 103203., 120105.,
107206., 103219., 108204., 107108., 114204., 128112., 126204.,
105106., 125103., 115206., 115221., 112211., 107223., 106205.,
102201., 121110., 105208., 111101., 112102., 107224., 124260.,
122202., 124210., 112212., 109200., 200486., 101213., 112243.,
201288., 200234., 104101., 117104., 201319., 126704.]

In [58]:
elementos = ['J121', 'J205', 'J210','J219', 'B974' ]
amal_df = pd.read_csv(path_data/'amal_df.csv').query('DIAG1 in @elementos').query('ESTAB not in @estbs_not')
real_df = pd.read_csv(path_data/'data.csv').drop(columns=['Unnamed: 0'])

C:\Users\ntrig\AppData\Local\Temp\ipykernel_21764\991073615.py:2: DtypeWarning: Columns (11,78,79,80,81,82,83,84) have mixed types. Specify dtype option on import or set low_memory=False.
  amal_df = pd.read_csv(path_data/'amal_df.csv').query('DIAG1 in @elementos').query('ESTAB not in @estbs_not')


In [59]:
amal_compare = amal_df[[col for col in real_df.columns if col in amal_df.columns]]
data_compare = real_df[[col for col in real_df.columns if col in amal_df.columns]]

In [63]:
data_compare[~data_compare.RUN.isin(amal_compare.RUN)].shape, data_compare.shape

((1560, 22), (39229, 22))

In [67]:
amal_compare.columns

Index(['RUN', 'fechaEgr', 'fechaIng', 'age', 'epiweek', 'year', 'season',
       'elegibilidad', 'ESTAB', 'DIAS_ESTAD', 'AREAF_1_TRAS', 'AREAF_2_TRAS',
       'AREAF_3_TRAS', 'AREAF_4_TRAS', 'AREAF_5_TRAS', 'AREAF_6_TRAS',
       'AREAF_7_TRAS', 'AREAF_8_TRAS', 'AREAF_9_TRAS', 'AREA_FUNC_I',
       'Macrozona', 'Macrozona2'],
      dtype='object')

In [70]:
amal_compare[~amal_compare.RUN.isin(data_compare.RUN)].epiweek.isna().sum()

0

In [69]:
amal_compare[~amal_compare.RUN.isin(data_compare.RUN)].dropna(subset=['elegibilidad']).shape, amal_compare.shape

((816, 22), (38280, 22))

In [40]:
ruts_comon = [run for run in amal_compare.RUN if run in data_compare.RUN]

data_compare = data_compare[data_compare.RUN.isin(ruts_comon)]
amal_compare = amal_compare[amal_compare.RUN.isin(ruts_comon)]


In [21]:
amal_compare.columns

Index(['RUN', 'fechaEgr', 'fechaIng', 'age', 'epiweek', 'year', 'season',
       'elegibilidad', 'ESTAB', 'DIAS_ESTAD', 'AREAF_1_TRAS', 'AREAF_2_TRAS',
       'AREAF_3_TRAS', 'AREAF_4_TRAS', 'AREAF_5_TRAS', 'AREAF_6_TRAS',
       'AREAF_7_TRAS', 'AREAF_8_TRAS', 'AREAF_9_TRAS', 'AREA_FUNC_I',
       'Macrozona', 'Macrozona2'],
      dtype='object')

In [41]:
data_compare.set_index('RUN').sort_values(by='RUN').compare(amal_compare.set_index('RUN').sort_values(by='RUN'))

RUN


In [7]:
real_df

,RUN,FECHA_NAC,fechaEgr,fechaIng,age,epiweek,year,season,elegibilidad,region,...,AREAF_5_TRAS,AREAF_6_TRAS,AREAF_7_TRAS,AREAF_8_TRAS,AREAF_9_TRAS,AREA_FUNC_I,Macrozona,Macrozona2,epiweekupc,VRS1
0,a45c240262a9ec36c6fb498cb4835fec4548d76d6c8d12...,2018-09-07,2019-08-28,2019-08-27,12,35,2019,in_season,2018.0,O'HIGGINS,...,0,0,0,0,0,0,Macrozona Centro Sur,Centro,-1,1
1,48bbeba41e725b2ffb387c38c4d18126c0268b5a3fe43a...,2019-03-22,2019-07-29,2019-07-21,4,29,2019,pre_season,2019.0,O'HIGGINS,...,0,0,0,0,0,1,Macrozona Centro Sur,Centro,29,1
2,f1855bf37d98d08f2d7d068cd39382c0a98f6255b0b93c...,2018-03-12,2019-07-29,2019-07-19,17,29,2019,pre_season,2018.0,O'HIGGINS,...,0,0,0,0,0,0,Macrozona Centro Sur,Centro,-1,1
3,fb764df83c3072fc23147b40b4730e63832b64250e8044...,2018-02-09,2019-08-28,2019-08-23,20,34,2019,pre_season,2018.0,ATACAMA,...,0,0,0,0,0,0,Macrozona Norte,Norte,-1,1
4,6774314587699f2c22cb51e597bbd9793447489cc92935...,2018-10-22,2019-07-29,2019-07-21,9,29,2019,pre_season,2019.0,O'HIGGINS,...,0,0,0,0,0,1,Macrozona Centro Sur,Centro,29,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
39224,7df9a455fd2a96ccd621b9978eb9cca298f959d7a1356d...,2023-11-20,2024-10-04,2024-09-21,10,38,2024,pre_season,2024.0,LOS LAGOS,...,0,0,0,0,0,0,Macrozona Sur,Sur,-1,1
39225,38b1ef66cac5afe45d6e54aa0ac95c7bf8056238b1725b...,2024-03-12,2024-09-30,2024-09-27,7,39,2024,pre_season,2024.0,BIOBIO,...,0,0,0,0,0,0,Macrozona Centro Sur,Centro,-1,1
39226,c9dba3122fdee9d1126778d5a109c0966499c22b07e84a...,2023-09-12,2024-06-24,2024-06-20,10,25,2024,in_season,2023.0,BIOBIO,...,0,0,0,0,0,0,Macrozona Centro Sur,Centro,-1,1
39227,5762f937baab9234df19f7ed8057cf64257b83a243f9c6...,2023-07-09,2024-11-08,2024-11-07,17,45,2024,in_season,2023.0,BIOBIO,...,0,0,0,0,0,0,Macrozona Centro Sur,Centro,-1,1


In [8]:
amal_df

,RUN,ESTAB,ServicioSalud,Seremi,SEXO,D_NAC,M_NAC,A_NAC,EDAD_CANT,TIPO_EDAD,...,diag_vrs_0,diag_vrs_1,diag_vrs_2,week,month,fechaEgr,duracion,elegible,elegibilidad2,diagnosis
0,560c59e419dc71a4a948b83449f776cb3d94df2d7b9cc1...,105107,5.0,NaN,NaN,15,6,2017,2,1,...,False,False,False,24,6,2019-06-25,4,0,0,2005
1,5d227f3aa6732dbb9f36bfafec9f2e1fff3ea5b81aa8c5...,112107,12.0,NaN,NaN,14,9,2018,9,2,...,False,False,True,26,7,2019-07-03,2,0,0,2063
2,a45c240262a9ec36c6fb498cb4835fec4548d76d6c8d12...,115110,15.0,NaN,NaN,7,9,2018,11,2,...,True,True,True,34,8,2019-08-28,1,0,0,2021
3,03196d46f41d82acd46222181b008683ac1c78d8637049...,115110,15.0,NaN,NaN,3,8,2018,9,2,...,False,False,False,20,5,2019-05-27,2,0,0,50
4,48bbeba41e725b2ffb387c38c4d18126c0268b5a3fe43a...,115110,15.0,NaN,NaN,22,3,2019,3,2,...,True,True,True,29,7,2019-07-29,8,1,1,2059
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
542433,621f2d7e5ea56eece15258f143cd1a842487e962b817f0...,109101,9.0,NaN,NaN,28,1,2022,2,1,...,False,False,False,25,6,2024-07-03,5,1,0,2023
542434,24773da7fc4c7992a1ce658bb807f30201c146f956ebaf...,109101,9.0,NaN,NaN,5,2,2022,2,1,...,False,False,False,28,7,2024-07-19,4,1,0,4175
542435,6aef5bf22b5c227c7af2fc40a9f1d081e3823cde8a21b4...,109101,9.0,NaN,NaN,21,3,2022,2,1,...,False,False,True,25,6,2024-06-25,1,1,0,2058
542436,649880c12a8863bc53d466d2cc0efb4f784d173728d6c0...,109101,9.0,NaN,NaN,15,12,2022,1,1,...,False,False,False,21,5,2024-06-01,1,1,0,210
